# Crossfire 四分类微调（Google Colab / T4）

从上往下依次运行每一格。训练**不需要** API key。

开始前：菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**。

结果会写到你的 Google Drive：`MyDrive/crossfire-runs/`。每个种子训完就同步一次，中途断线也不至于全丢。

## 1. 检查有没有 GPU

打印 `nvidia-smi`。如果报错或没有 GPU，说明运行时还是 CPU——去改成 T4，然后**停止并重新连接**，再从这一格重跑。

大概几秒钟。

In [ ]:
import subprocess, sys
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    print("\n❌ 没有 GPU。请：代码执行程序 → 更改运行时类型 → T4 GPU，然后重新连接运行时。")
    sys.exit("No GPU")
print("✅ 看到 GPU 信息就可以继续下一格。")

## 2. 挂载 Google Drive

把结果存进云盘，避免 Colab 会话断开后文件消失。首次会弹出授权窗口，点允许即可。

会在 Drive 里建 `crossfire-runs/`。大约半分钟（含授权）。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/crossfire-runs")
RUNS.mkdir(parents=True, exist_ok=True)
print("结果目录：", RUNS)

## 3. 克隆仓库并安装训练依赖

用 `uv sync --group train`，版本和本地 `pyproject.toml` 一致（torch / transformers 等）。

首次大约 3–8 分钟（下载 torch 较大）。

In [ ]:
%%bash
set -euo pipefail
cd /content
rm -rf Crossfire
git clone --depth 1 https://github.com/FishySean/Crossfire.git
cd Crossfire
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --group train
wc -l data/pairs.jsonl

## 4. 自检：CUDA、显卡型号、是否支持 bf16

`torch.cuda.is_available()` 必须是 True。T4 通常 **不支持 bf16**，脚本会自动改用 **fp16 + GradScaler**（参数仍是 fp32，安全）。

这一格用 `%%bash` + heredoc（不要用 `!` 包 heredoc，IPython 只会执行第一行）。

几秒钟。

In [ ]:
%%bash
cd /content/Crossfire
export PATH="$HOME/.local/bin:$PATH"
# 必须拉最新代码：bf16 按算力版本判断，不再信 is_bf16_supported 的默认模拟
git pull --ff-only origin main
uv run python - <<'PY'
import torch
from training.amp import choose_amp, describe_device

assert torch.cuda.is_available(), "CUDA 不可用：请换成 T4 GPU 运行时"
device = torch.device("cuda")
cap = torch.cuda.get_device_capability(device)
amp = choose_amp(device)
print("显卡：", describe_device(device))
print("算力版本：", f"{cap[0]}.{cap[1]}")
print("is_bf16_supported() 默认：", torch.cuda.is_bf16_supported())  # T4 可能仍是 True（模拟）
print("将使用：", amp.name)
print("原因：", amp.reason)
expected = "fp16+scaler" if cap[0] < 8 else "bf16"
assert amp.name == expected, f"算力 {cap} 应选 {expected}，实际 {amp.name}"
PY

## 5. GPU 冒烟（--smoke）

只跑 1 个种子、很少的 batch，确认训练循环、存盘、画图都通。结束会打印**峰值显存**。

大约 1–3 分钟。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
SMOKE_MIRROR="/content/drive/MyDrive/crossfire-runs/smoke"
mkdir -p "$SMOKE_MIRROR"
uv run python -m training.train --smoke --mirror-dir "$SMOKE_MIRROR"

## 6. 正式训练（三个种子）

种子 42 / 43 / 44。早停看验证集加权损失。每个种子结束后立刻同步到 Drive。

T4 上大约 **15–25 分钟**。可以去喝水，但别关标签页太久（闲置可能断连；有 Drive 同步可兜底）。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
stamp=$(date -u +%Y%m%d-%H%M%S)
MIRROR="/content/drive/MyDrive/crossfire-runs/run-${stamp}"
mkdir -p "$MIRROR"
echo "Drive 同步目录：$MIRROR"
uv run python -m training.train --mirror-dir "$MIRROR"

## 7. 查看打包结果

打包里有：fp16 最终模型、三个种子的报告和曲线、summary。**没有**其它种子的大 checkpoint，也没有优化器状态。

在 Drive 的 `crossfire-runs/run-.../package/` 里下载即可。

In [ ]:
from pathlib import Path
import json

mirror = Path("/content/drive/MyDrive/crossfire-runs")
runs = sorted(mirror.glob("run-*"), key=lambda p: p.stat().st_mtime)
assert runs, "还没有 run-* 目录，请先跑完正式训练那一格"
latest = runs[-1]
pack = latest / "package"
print("最新运行：", latest)
print("打包目录：", pack)
if (pack / "pack_meta.json").exists():
    print(json.dumps(json.loads((pack / "pack_meta.json").read_text()), indent=2, ensure_ascii=False))
if (latest / "summary.json").exists():
    s = json.loads((latest / "summary.json").read_text())
    print("入选种子：", s.get("selected_seed_by_val_loss"))
    print("三种子测试宏 F1：", s.get("test_macro_f1_by_seed"))
    print("汇总：", s.get("test_macro_f1_summary"))

## 8. 第 10 步：配置选择（分组 5 折 CV）

**测试集全程不要动**——下面命令都只看 train 池的 CV macro-F1。
一次只改一样；每跑完一格把 `cv_summary.json` 里的均值记进 `notes/iteration-2.md`。

顺序：
1. **a** 基线（新数据 + 默认 epochs=12 / balanced）
2. **b** epochs 25 + early stopping（patience 保持 3）
3. **c** `--class-weight-scheme sqrt`（在 a/b 胜者上改）
4. **d** 胜者再加 `--calibrate-temperature`（只报告 ECE，不改 CV 选型）

每折大约几分钟；5 折合计大约 **20–40 分钟**（视配置）。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
git pull --ff-only origin main
BASE="/content/drive/MyDrive/crossfire-runs/iter2"
mkdir -p "$BASE"

echo "===== a) baseline CV ====="
uv run python -m training.train --cv 5 \
  --output-dir out/training/iter2-a \
  --mirror-dir "$BASE/a-baseline"

echo "===== b) epochs 25 CV ====="
uv run python -m training.train --cv 5 --epochs 25 \
  --output-dir out/training/iter2-b \
  --mirror-dir "$BASE/b-epochs25"

echo "===== c) sqrt class weights CV ====="
# 若 b 的 CV 均值更高，把 --epochs 25 留着；否则删掉 --epochs 25
uv run python -m training.train --cv 5 --epochs 25 \
  --class-weight-scheme sqrt \
  --output-dir out/training/iter2-c \
  --mirror-dir "$BASE/c-sqrt"

echo "===== d) temperature scaling on winner CV ====="
uv run python -m training.train --cv 5 --epochs 25 \
  --class-weight-scheme sqrt \
  --calibrate-temperature \
  --output-dir out/training/iter2-d \
  --mirror-dir "$BASE/d-temp"

python - <<'PY'
import json
from pathlib import Path
base = Path('/content/drive/MyDrive/crossfire-runs/iter2')
for name in ['a-baseline','b-epochs25','c-sqrt','d-temp']:
    p = base / name / 'cv_summary.json'
    if not p.exists():
        print(name, 'MISSING')
        continue
    s = json.loads(p.read_text())
    print(f"{name}: CV macro-F1 {s['cv_macro_f1_mean']:.4f} ± {s['cv_macro_f1_std']:.4f}  scheme={s.get('class_weight_scheme')} epochs={s.get('max_epochs')}")
    if s.get('fold_scores') and s['fold_scores'][0].get('calibration'):
        cals = [f['calibration'] for f in s['fold_scores'] if 'calibration' in f]
        if cals:
            before = sum(c['ece_before'] for c in cals)/len(cals)
            after = sum(c['ece_after'] for c in cals)/len(cals)
            print(f"  ECE {before:.4f} → {after:.4f}")
PY

## 9. 第 10 步：最终三种子（只用一次测试集）

把下面命令里的 `--epochs` / `--class-weight-scheme` 换成 **CV 胜者**。  
三种子训完才看测试集；把 `summary.json` / `test_report.md` 下载回来给我写 `notes/iteration-2.md`。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
stamp=$(date -u +%Y%m%d-%H%M%S)
MIRROR="/content/drive/MyDrive/crossfire-runs/iter2-final-${stamp}"
mkdir -p "$MIRROR"
echo "Drive 同步目录：$MIRROR"
# TODO: 按 CV 胜者改 epochs / class-weight-scheme；需要校准则加 --calibrate-temperature
uv run python -m training.train \
  --epochs 25 \
  --class-weight-scheme sqrt \
  --calibrate-temperature \
  --mirror-dir "$MIRROR"
ls -lah "$MIRROR/package" | head

## 10. 第 10b 步：屏蔽 unrelated 评估（不训练）

在 Drive 上的 **v1**（`run-20261121-235427`）和最新完整 **v2**（`iter2-final-*`）各 3 个种子的 `best_checkpoint` 上，对**冻结测试集**分别跑开/关屏蔽。

- 需要先挂载 Drive（§2）
- 本格会切换到 GitHub 分支 **`iter2-mask`**（含 `NLI_MASK_UNRELATED` / `training.evaluate --compare-mask`）
- 结果写到 Drive：`crossfire-runs/iter2-mask-eval/`
- 找不到目录会报错并列出 `crossfire-runs/` 下实际文件夹

**从下面两格开始点**（已挂载 Drive 的话）：先跑「检出 iter2-mask」，再跑「评估」。约数分钟（CPU/GPU 均可）。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content
if [ ! -d Crossfire/.git ]; then
  git clone --branch iter2-mask --single-branch https://github.com/FishySean/Crossfire.git
else
  cd Crossfire
  git remote set-url origin https://github.com/FishySean/Crossfire.git
  git fetch origin iter2-mask
  git checkout iter2-mask
  git pull --ff-only origin iter2-mask
  cd /content
fi
cd Crossfire
# 已在分支上时再确认一次
git fetch origin iter2-mask
git checkout iter2-mask
git pull --ff-only origin iter2-mask
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --group train
echo "branch=$(git rev-parse --abbrev-ref HEAD)  commit=$(git rev-parse --short HEAD)"

In [ ]:
from pathlib import Path
import traceback

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

import sys
sys.path.insert(0, "/content/Crossfire")

from training.mask_unrelated_eval import run_colab_eval

RUNS = Path("/content/drive/MyDrive/crossfire-runs")
PAIRS = Path("/content/Crossfire/data/pairs.jsonl")
OUT = RUNS / "iter2-mask-eval"

print("listing crossfire-runs/:", sorted(p.name for p in RUNS.iterdir()) if RUNS.is_dir() else "(missing)")
try:
    payload = run_colab_eval(runs_root=RUNS, pairs_path=PAIRS, out_dir=OUT)
except FileNotFoundError as exc:
    print("\n*** 目录发现失败 ***")
    print(exc)
    raise
except Exception:
    traceback.print_exc()
    raise

print("publish_gate_ok =", payload["publish_gate_ok"])
print("把 iter2-mask-eval/ 里的 slim JSON 发回本地对话即可继续发布判定。")

## 做完了

从 Google Drive 下载 `package/` 文件夹。Colab 可以断开；**没有**像 RunPod 那样的磁盘持续计费。

若还开着其它付费云主机（算力自由 / RunPod），请登录控制台确认已 **删除 / Terminate**，避免继续扣费。